In [12]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import re
pd.options.mode.chained_assignment = None

In [13]:

#! Load data from file
df = pd.read_csv('raw_data/bayut_listings.csv')
#* Grab the numeric columns
numeric_cols = ['Price', 'Number Of Bedrooms', 'Number Of Bathrooms', 'Area']
#* Grab categorical columns
categorical_cols = df.drop(numeric_cols,axis=1).columns
#* total number of rows
data_count = df.count().iloc[0]
#* 1% threshold that will be used to cut off low frequency classes
threshold = data_count // 100
df.head()

,Price,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,"1,650,000",Floor,5,4,523 Sq. M.,"Al Rayyan, East Riyadh, Riyadh"
1,"1,249,000",Floor,6,5,500 Sq. M.,"Al Khaleej, East Riyadh, Riyadh"
2,"499,000",Apartment,3,3,154 Sq. M.,"Al Sakb, Madina"
3,"499,000",Apartment,3,3,154 Sq. M.,"Al Sakb, Madina"
4,"499,000",Apartment,3,3,154 Sq. M.,"Al Sakb, Madina"


In [14]:
df2 = df.drop_duplicates()

In [16]:

#? function that is used filter non numeric characters in numeric features if value is completely non numeric replace it with NAN
def clean_numeric(x):
    digits = re.sub(r'\D', '', str(x))
    return digits if digits else np.nan


In [15]:

#? function for cleaning the housing prices data in dataframe
def data_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    #* drop rows containing empty values
    df.dropna(inplace=True)
    #* standardize categorical to be in upper case
    df = df.map(lambda x: str(x).upper())
    #* filter non english characters from "property type" column
    df['Property Type'] = df['Property Type'].astype(str).map(lambda x: re.sub(r'[^A-Za-z\s]', '', x))

    #* clean non-numeric characters from the numeric columns and then casting their type to int
    df[numeric_cols] = df[numeric_cols].map(clean_numeric).astype('Int64')
    #* drop rows containing empty values which is caused by the clean_numeric function
    df.dropna(inplace=True)
    #* filter non english characters from "Location" column while still retaining punctuation characters
    df['Location'] = df['Location'].astype(str).map(lambda x: re.sub(r'[^A-Za-z,.\s]', '', x))
    #* remove property types that have frequency less than the threshold
    df = df.groupby('Property Type').filter(lambda x : len(x)> threshold)
    #* group Locations that have frequency less than the threshold to one value called OTHER
    location_counts = df['Location'].value_counts()
    frequent_locations = location_counts[location_counts >= threshold].index
    df['Location'] = df['Location'].apply(lambda x: x if x in frequent_locations else 'OTHER')
    #* reset index 
    df = df.reset_index().drop(columns='index',axis=1)

    return df

In [17]:
data_duplicated = data_cleaning(df)
data_nonDuplicated = data_cleaning(df2)
#! Save clean data to csv file
data_nonDuplicated.to_csv('bayut_listings_cleaned_notDuplicated.csv',index=False)

data_duplicated.to_csv('bayut_listings_cleaned.csv',index=False)





